In [ ]:
from dotenv import load_dotenv

load_dotenv()


In [ ]:
import os
from openai import OpenAI
import requests

client = OpenAI(
    api_key=os.getenv("VCS_LLM_API_KEY"),
    base_url=os.getenv("VCS_LLM_API_KEY")
)

messages = [
    {
        "role": "user",
        "content": "Cần tìm hiểu về danh sách sản phẩm của VCS - Trung tâm an ninh mạng viettel"
    }
]
temperature = 0.2

response = client.chat.completions.create(
                model = None,
                messages = messages,
                temperature=temperature,
                stream=False
            )

print(response.choices[0].message.content)


In [ ]:
import json
from openai import OpenAI

client = OpenAI(api_key="OPENAI_API_KEY")

MODEL = "gpt-4o-mini"  # hoặc model bạn đang dùng


SEMANTIC_PROMPT = """
You are a senior analytics engineer.

Task:
- Convert database schema into a semantic layer for Text-to-SQL.
- ONLY use provided schema.
- DO NOT invent tables, columns, metrics.
- If unsure, leave field empty.

Rules:
- Classify table as: fact | dimension | bridge
- Detect:
  - primary_key
  - time_column (created_at, date, timestamp)
  - metrics (numeric, aggregatable)
  - dimensions (categorical, id)
- Infer joins ONLY if column name ends with _id

Return JSON only.
No explanation.
"""


def load_schema(path="data_catalog.json"):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def call_llm(schema):
    response = client.chat.completions.create(
        model=MODEL,
        temperature=0,
        messages=[
            {"role": "system", "content": SEMANTIC_PROMPT},
            {
                "role": "user",
                "content": json.dumps(schema, ensure_ascii=False)
            }
        ],
    )

    content = response.choices[0].message.content
    return json.loads(content)


def save_semantic(data, path="semantic_catalog.json"):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)


def main():
    schema = load_schema()
    semantic = call_llm(schema)
    save_semantic(semantic)

    print("✅ semantic_catalog.json generated")

In [ ]:

EMBEDDING_URL=os.getenv("VCS_EMBBEDING_URL")
from typing import List
import json
def get_embedding(text: str) -> List[float]:

    payload = {
        "input": text
    }
    headers = {
        "Content-Type": "application/json"
    }
    
    try:
        response = requests.post(
            EMBEDDING_URL,
            headers=headers,
            data=json.dumps(payload),
            verify=False,
            timeout=10
        )
        response.raise_for_status()
        embedding = response.json()["data"][0]["embedding"]
        return embedding
    except Exception as e:
        print(f"Error getting embedding: {e}")
        raise
    
get_embedding("hello viet name")